In [1]:
# script to count instances of motifs overlapping the variants in hits vs. misses #

In [1]:
# import packages #
import pandas as pd
import numpy as np
from collections import Counter

In [2]:
# define function to count motif hits overlapping the variant #
def count_var_overlap (path2fimo_out):
    # open fimo output
    fimo_output = pd.read_csv(path2fimo_out, sep = '\t', comment='#')
    n_seqs = len(fimo_output['sequence_name'].unique())
    # convert floats to ints
    fimo_output['start'] = [int(i) for i in fimo_output['start']]
    fimo_output['stop'] = [int(i) for i in fimo_output['stop']]
    # print length of hits
    print(f'There are {len(fimo_output)} motif matches in {n_seqs} sequences')
    # add whether the variant overlaps a motif #
    fimo_output['overlap_var'] = [1 if 100 in range(start, stop) else 0 for start, stop in zip(fimo_output['start'], fimo_output['stop'])]
    # add a column with the ref/alt dropped
    fimo_output['var_id'] = [(':').join(i.split(':')[0:-2]) for i in fimo_output['sequence_name']]
    # count the number of sequences that contain a variant overlapping the motif
    n_overlap = len(fimo_output[fimo_output['overlap_var'] == 1]['sequence_name'].unique())
    print(f'There are {n_overlap} sequences with a variant overlapping a motif')
    return fimo_output

In [50]:
# define a function for opening missed predictions and returning list of IDs with alt IDs for a given cell type
def return_missed_pred_ids (path2missed_preds,
                           cell_type):
    # open full missed preds
    all_missed = pd.read_csv(path2missed_preds, sep = '\t')
    # get labels for the chosen cell type
    cell_labels = [i for i in all_missed['missed_cell_type'].unique() if cell_type in i]
    # filter all missed predictions for associated cell type
    cell_misses = all_missed[all_missed['missed_cell_type'].isin(['K562', 'K562:HEPG2', 'K562:SKNSH', 'K562:HEPG2:SKNSH'])]
    # make list of miss ids
    miss_refs = list(cell_misses['hg19_id'])
    # add alt sequence IDs
    miss_alts = [(':').join([i.split(':')[0], 
                           i.split(':')[1],
                           i.split(':')[2], 
                           i.split(':')[3],
                           'A', 
                           i.split(':')[-1]]) for i in miss_refs]
    # combine into one
    all_miss_ids = miss_refs + miss_alts
    return all_miss_ids

In [51]:
# get IDs of all misses for each cell type
# K562
k_misses = return_missed_pred_ids('data/all_emvars_missed_predictions_only.tsv',
                                  'K562')
# HepG2
h_misses = return_missed_pred_ids('data/all_emvars_missed_predictions_only.tsv',
                                  'HEPG2')
# SKNSH
s_misses = return_missed_pred_ids('data/all_emvars_missed_predictions_only.tsv',
                                  'SKNSH')

In [118]:
# define function for opening fimo and adding start, stop, filtering missed predictions, 
# and returning only significant hits overlapping the variant
def open_fimo (path2fimo,
               missed_preds_list,
               qvalue_threshold):
    # open fimo
    fimo_all = pd.read_csv(path2fimo, sep = '\t', comment = '#')
    # convert start/stop to ints for calling whether motif variant overlaps the variant
    fimo_all['start'] = [int(i) for i in fimo_all['start']]
    fimo_all['stop'] = [int(i) for i in fimo_all['stop']]
    # add variant ID without Ref Alt
    fimo_all['var_id'] = [(':').join(i.split(':')[0:-2]) for i in fimo_all['sequence_name']]
    # add whether the variant overlaps a motif
    fimo_all['overlap_var'] = [1 if 100 in range(start, stop + 1) else 0 for start, stop in zip(fimo_all['start'], 
                                                                                                fimo_all['stop'])]
    # filter for only motif hits that overlap the variant
    # are not in the list of missed IDs
    # have a q-value below the provided threshold
    fimo_sig_hits = fimo_all[~(fimo_all['sequence_name'].isin(missed_preds_list)) &
                             (fimo_all['overlap_var'] == 1) &
                             (fimo_all['q-value'] < qvalue_threshold)]
    return fimo_sig_hits.sort_values(by='sequence_name')

In [124]:
# open and filter fimo hits for each cell type
# K562
k562_sig_fimo = open_fimo('k562_95033/k562_all_emvars_fimo/fimo.tsv',
                          k_misses,
                          .01)
# HepG2
hepg2_sig_fimo = open_fimo('hepg2_95033/hepg2_all_emvars_fimo/fimo.tsv',
                           h_misses,
                           .01)
# SKNSH
sknsh_sig_fimo = open_fimo('sknsh_95033/sknsh_all_emvars_fimo/fimo.tsv',
                           s_misses,
                           .01)

In [23]:
# open all emvars
# k562
k562_emvars = pd.read_csv('data/k562_emvars_with_controls.tsv', sep = '\t', low_memory=False)
# hepg2
hepg2_emvars = pd.read_csv('data/hepg2_emvars_with_controls.tsv', sep = '\t', low_memory=False)
# sknsh
sknsh_emvars = pd.read_csv('data/sknsh_emvars_with_controls.tsv', sep = '\t', low_memory=False)

In [165]:
k_check = k562_emvars.copy()
k_check['k_mpac_emvar'] = [1 if abs(i) > .05 else 0 for i in k_check['k562_skew_pred_avg']]

In [167]:
from scipy import stats

In [168]:
stats.pearsonr(k_check['Log2Skew'], k_check['k562_skew_pred_avg'])

PearsonRResult(statistic=0.7108252670023593, pvalue=0.0)

In [151]:
# define a function for calculating the percentage of sequences containing a significant motif hit overlapping the variant
def percent_seqs_with_hit (sig_fimo_hits,
                           emvar_df,
                           miss_ids,
                           cell_type):
    # print the cell type to start the report
    print(f'{cell_type.upper()}: Missed emVar FIMO Match Summary')
    # filter emvar df for missed emvar ids
    dropped_misses = emvar_df[~emvar_df['ID'].isin(miss_ids)]
    # double the length of the dropped miss df to account for both ref and alt
    n_emvar_seqs = len(dropped_misses) * 2
    # iterate through motifs individually and calculate percent of all emVar sequences have that hit
    for motif in sig_fimo_hits['motif_id'].unique():
        motif_df = sig_fimo_hits[sig_fimo_hits['motif_id'] == motif]
        # get the number of sequences with a match for that motif
        n_motif = len(motif_df['sequence_name'].unique())
        # calculate that as a percent of all sequences
        percent_motif = (n_motif / n_emvar_seqs) * 100
        round_percent = round(percent_motif,2)
        # print a little report
        print(motif)
        print(f'{round_percent}% of {cell_type} emVars have a significant match for this motif.')

In [156]:
# Calculate the percentage of sequences with a motif match
# K562
percent_seqs_with_hit(k562_sig_fimo,
                      k562_emvars,
                      k_misses,
                      'K562')

K562: Missed emVar FIMO Match Summary
pattern_0
5.83% of K562 emVars have a significant match for this motif.
pattern_3
0.94% of K562 emVars have a significant match for this motif.
pattern_2
0.9% of K562 emVars have a significant match for this motif.
pattern_5
0.18% of K562 emVars have a significant match for this motif.
pattern_6
0.0% of K562 emVars have a significant match for this motif.


In [154]:
# Calculate the percentage of sequences with a motif match
# HepG2
percent_seqs_with_hit(hepg2_sig_fimo,
                      hepg2_emvars,
                      h_misses,
                      'HepG2')

HEPG2: Missed emVar FIMO Match Summary
pattern_2
1.81% of HepG2 emVars have a significant match for this motif.
pattern_1
0.43% of HepG2 emVars have a significant match for this motif.
pattern_3
1.54% of HepG2 emVars have a significant match for this motif.


In [155]:
# Calculate the percentage of sequences with a motif match
# SKNSH
percent_seqs_with_hit(sknsh_sig_fimo,
                      sknsh_emvars,
                      s_misses,
                      'SK-N-SH')

SK-N-SH: Missed emVar FIMO Match Summary
pattern_2
3.79% of SK-N-SH emVars have a significant match for this motif.
pattern_0
0.48% of SK-N-SH emVars have a significant match for this motif.
pattern_4
0.05% of SK-N-SH emVars have a significant match for this motif.
